# Calibration methods for OCR activations

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/Document-OCR-Pipeline/blob/main/colab/02_ocr_pipeline_quant_C.ipynb)

**Open in Google Colab:** https://colab.research.google.com/github/Gaurav14cs17/Document-OCR-Pipeline/blob/main/colab/02_ocr_pipeline_quant_C.ipynb

This notebook collects **scales** from Florence-2 while it runs real OCR. It does not round weights and it does not implement a quantizer.

After `forward()`, layer inputs and outputs are gone. A **hook** catches them. An **observer** turns the caught tensor into a scale (and a zero-point if the range is not centered at 0). That is calibration.

## 6 observers, 3 attach points, 2 granularities

The hook path is always the same. Only the observer changes.

### Attach points

| # | Tensor | How you get it | Hook? |
|---|--------|----------------|-------|
| 1 | Layer input `X` | `register_forward_hook` → `inp[0]` | Yes |
| 2 | Layer output `Y` | `register_forward_hook` → `out` | Yes |
| 3 | Weight `W` | `layer.weight` | No |

### Observers

| # | Observer | Tracks | Scale |
|---|----------|--------|-------|
| 1 | MinMax | running abs-max | `s = absmax / qmax` |
| 2 | Affine MinMax | running min and max | `s = (max - min) / (2^b - 1)` and a zero-point |
| 3 | EMA MinMax | abs-max with momentum | same as MinMax, smoothed across batches |
| 4 | Percentile | p-th quantile of abs values | `s = quantile / qmax` |
| 5 | Histogram + KL | histogram of values | clip whose quantized hist is closest (KL) |
| 6 | MSE | grid of clip ratios | `s` with lowest reconstruction error |

`qmax = 2^(b-1) - 1` (int8 → 127).

### Granularity

| Kind | Meaning |
|------|---------|
| Per-tensor | one scale for the whole tensor |
| Per-channel | one scale per feature (last dim of a Linear input) |

## Run order

```
Install → Load Florence-2 + document
       → Hook a Linear layer → generate (OCR)
       → observer.update(X) → remove hook → finish() → scale
       → fake-quant X to score the scale → compare observers
```


## Step 0 — Install and config

Pins `numpy==2.1.3`, `scipy==1.14.1`, `scikit-learn==1.6.1`, `transformers==4.49.0`. Use a GPU if you have one.

| Setting | Default | Role |
|---------|---------|------|
| `MAX_CALIB_BATCHES` | 8 | OCR `generate` passes while the hook is on |
| `BITS_ACT` | 8 | Bit-width used only to score the scale |
| `PERCENTILE` | 99.99 | Quantile for the percentile observer |
| `EMA_MOMENTUM` | 0.95 | EMA keeps 95% of the previous abs-max |
| `HIST_BINS` | 2048 | Histogram bins before KL search |
| `MSE_GRID` | 20 | Clip ratios the MSE observer tries |
| `SAMPLE_CAP` | 50000 | Max tokens kept for percentile, KL, and MSE |


In [ ]:
import os, re, subprocess, sys

MODEL_ID = "microsoft/Florence-2-base-ft"
OCR_PROMPT = "<OCR_WITH_REGION>"
MAX_CALIB_BATCHES = 8
BITS_ACT = 8
PERCENTILE = 99.99
EMA_MOMENTUM = 0.95
HIST_BINS = 2048
MSE_GRID = 20
MIN_PARAMS = 4096
ALWAYS_SKIP_PATTERNS = ("lm_head", "embed", "vision", "patch_embed")
SAMPLE_CAP = 50_000


def _pip_version(pkg):
    r = subprocess.run([sys.executable, "-m", "pip", "show", pkg],
                       capture_output=True, text=True, check=False)
    m = re.search(r"^Version: (.+)$", r.stdout, re.M)
    return m.group(1) if m else ""


def ensure_numpy_stack():
    want = {"numpy": "2.1.3", "scipy": "1.14.1", "scikit-learn": "1.6.1"}
    if any(_pip_version(p) != v for p, v in want.items()):
        pkgs = [f"{p}=={v}" for p, v in want.items()]
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "--force-reinstall", *pkgs])
    import numpy as np
    import scipy
    import sklearn
    print(f"numpy {np.__version__} | scipy {scipy.__version__} | sklearn {sklearn.__version__}")


def _transformers_version_ok(version: str) -> bool:
    return version.startswith("4.49")


def ensure_transformers():
    ensure_numpy_stack()
    ver = _pip_version("transformers")
    if not _transformers_version_ok(ver):
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "transformers==4.49.0"])
        ver = _pip_version("transformers")
    import transformers
    if not _transformers_version_ok(transformers.__version__):
        print("Restart runtime, then Run all."); os.kill(os.getpid(), 9)
    return ver


ensure_numpy_stack()
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "torch", "accelerate", "pillow", "matplotlib", "requests", "huggingface_hub"])
print(f"Ready — transformers {ensure_transformers()}")


## Step 1 — Hooks

Use **recorder classes** (same pattern as `02_ocr_pipeline_quant_1A`):

| When | What |
|------|------|
| **Setup** | `register_layer_hooks(model, recorders)` |
| **Forward** | `run()` / `generate` → `recorder.__call__` |
| **Cleanup** | `remove_hooks(handles)` |

| Class | Role |
|-------|------|
| `ObserverUpdateRecorder` | `observer.update(X)` for scale calibration |
| `LayerCaptureRecorder` | save $X_l$ batches to a dict |

```
1. recorder = ObserverUpdateRecorder(obs, where="input")
2. handles = register_layer_hooks(model, {layer_name: recorder})
3. cal.run(n_batches)   # hooks fire here
4. remove_hooks(handles)
5. scale = observer.finish()
```


In [ ]:
from __future__ import annotations
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from io import BytesIO

import matplotlib.pyplot as plt
import requests
import torch
import torch.nn as nn
from PIL import Image, ImageDraw

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")


In [ ]:
class LinearLayerScanner:
    def __init__(self, root: nn.Module):
        self.root = root

    def layers(self, limit=None):
        out = [(n, m) for n, m in self.root.named_modules() if isinstance(m, nn.Linear)]
        return out[:limit] if limit else out

    def is_skipped(self, name: str) -> bool:
        n = name.lower()
        return any(p in n for p in ALWAYS_SKIP_PATTERNS)


class ImagePadder:
    @staticmethod
    def pad(image):
        w, h = image.size
        side = max(w, h)
        c = Image.new("RGB", (side, side), "white")
        px, py = (side - w) // 2, (side - h) // 2
        c.paste(image, (px, py))
        return c, px, py, w, h



def register_layer_hooks(model, recorders: dict):
    """① SETUP — hook body runs later during run()/generate."""
    handles = []
    for layer_name, recorder in recorders.items():
        handles.append(model.get_submodule(layer_name).register_forward_hook(recorder))
    return handles


def remove_hooks(handles):
    """Cleanup — detach all hooks."""
    for h in handles:
        h.remove()


class ObserverUpdateRecorder:
    """Hook: call observer.update(X) from layer input or output."""

    def __init__(self, observer, where="input"):
        self.observer = observer
        self.where = where  # "input" or "output"

    def __call__(self, module, inputs, output):
        src = inputs[0] if self.where == "input" else output
        x = src[0] if isinstance(src, tuple) else src
        if x is not None:
            self.observer.update(x.detach())


class LayerCaptureRecorder:
    """Hook: save X_l (or Y) batches for offline analysis."""

    def __init__(self, layer_name, storage, where="input"):
        self.layer_name = layer_name
        self.storage = storage
        self.where = where

    def __call__(self, module, inputs, output):
        src = inputs[0] if self.where == "input" else output
        x = src[0] if isinstance(src, tuple) else src
        if x is not None:
            self.storage[self.layer_name].append(
                x.detach().reshape(-1, x.shape[-1]).cpu()
            )

class HookCalibrator:
    def __init__(self, model, processor, image, prompt=OCR_PROMPT, device=DEVICE):
        self.model, self.processor, self.image = model, processor, image
        self.prompt, self.device = prompt, device

    def _padded(self):
        canvas, *_ = ImagePadder.pad(self.image)
        return canvas

    def run(self, n_batches: int):
        padded = self._padded()
        for _ in range(n_batches):
            inp = self.processor(text=self.prompt, images=padded, return_tensors="pt").to(self.device)
            inp["pixel_values"] = inp["pixel_values"].to(dtype=next(self.model.parameters()).dtype)
            with torch.no_grad():
                self.model.generate(input_ids=inp["input_ids"], pixel_values=inp["pixel_values"],
                                    max_new_tokens=64, do_sample=False, num_beams=1)

    def attach(self, layer_names, observers: dict, where: str = "input"):
        """② FORWARD happens in run() after you call attach()."""
        recorders = {
            name: ObserverUpdateRecorder(observers[name], where)
            for name in layer_names
        }
        return register_layer_hooks(self.model, recorders)

    def collect(self, layer_names, n_batches, where: str = "input"):
        store = {n: [] for n in layer_names}
        recorders = {
            n: LayerCaptureRecorder(n, store, where) for n in layer_names
        }
        handles = register_layer_hooks(self.model, recorders)
        self.run(n_batches)
        remove_hooks(handles)
        return {n: torch.cat(v, 0) if v else None for n, v in store.items()}

print("LinearLayerScanner + HookCalibrator OK")

## Step 2 — Load the model and one document page

The hook must see the same kind of activations you get at OCR time. Load Florence-2, then a sample page. Later cells call `generate` with `<OCR_WITH_REGION>`.


In [ ]:
# Load Florence-2
ensure_transformers()
from transformers import AutoProcessor, AutoModelForCausalLM


class FlorenceModelLoader:
    def __init__(self, model_id=MODEL_ID, device=DEVICE):
        self.model_id, self.device = model_id, device
        self.dtype = torch.float16 if device == "cuda" else torch.float32

    def load_processor(self):
        return AutoProcessor.from_pretrained(self.model_id, trust_remote_code=True)

    def load_model(self):
        m = AutoModelForCausalLM.from_pretrained(
            self.model_id, trust_remote_code=True,
            torch_dtype=self.dtype, attn_implementation="eager").to(self.device)
        m.eval()
        return m

print("FlorenceModelLoader OK")


In [ ]:
_loader = FlorenceModelLoader()
processor = _loader.load_processor()
model = _loader.load_model()
dtype = _loader.dtype

try:
    url = "https://raw.githubusercontent.com/Gaurav14cs17/Document-OCR-Pipeline/main/assets/table_page.png"
    image = Image.open(BytesIO(requests.get(url, timeout=30).content)).convert("RGB")
except Exception:
    image = Image.new("RGB", (640, 480), "white")
    ImageDraw.Draw(image).text((20, 20), "Sample", fill="black")

print(f"Loaded {MODEL_ID} — {sum(p.numel() for p in model.parameters())/1e6:.1f}M params")
print(f"Calibration image: {image.size[0]}×{image.size[1]} px")
plt.figure(figsize=(6, 4)); plt.imshow(image); plt.title("Calibration page (real OCR distribution)")
plt.axis("off"); plt.show()

scanner = LinearLayerScanner(model)
linear_layers = scanner.layers()
print(f"nn.Linear count: {len(linear_layers)}")


## Step 3 — The 6 observers

Every observer has the same two calls:

```
update(x)   # fold this batch into a running statistic
finish()    # return CalibState(scale, zero_point, method)
```

Fake-quant is only how we **score** a scale: round `X` with `s` and measure how far `Q(X)` is from `X`.

**Symmetric** (zero stays at 0). `qmax = 2^(b-1) - 1`:

$$
q = \mathrm{clip}\big(\mathrm{round}(x / s),\; -q_{\max}-1,\; q_{\max}\big), \qquad \hat{x} = q \cdot s
$$

**Affine** (range need not be centered). `z` is the zero-point:

$$
s = \frac{x_{\max} - x_{\min}}{2^{b}-1}, \qquad
z = \mathrm{round}(-x_{\min} / s), \qquad
\hat{x} = s \cdot \big(\mathrm{clip}(\mathrm{round}(x/s)+z) - z\big)
$$

| Observer | Outliers | `finish()` returns | Use when |
|----------|----------|--------------------|----------|
| MinMax | kept (they inflate `s`) | one scale | baseline |
| Affine MinMax | kept, range can sit off zero | scale and zero-point | `Y` that is not symmetric |
| EMA MinMax | smoothed | one scale | many decode steps |
| Percentile | values above `p` dropped | one scale | rare spikes |
| Histogram + KL | clip from KL search | one scale | histogram calibrator |
| MSE | clip from reconstruction | one scale | you can store a sample of `X` |

Weights use MinMax on `layer.weight`. No hook.


In [ ]:
def qmax_sym(n_bits: int) -> int:
    return 2 ** (n_bits - 1) - 1


def fake_quant_sym(x: torch.Tensor, scale: torch.Tensor, n_bits: int) -> torch.Tensor:
    qm = qmax_sym(n_bits)
    s = scale.clamp(min=1e-8)
    q = (x / s).round().clamp(-qm - 1, qm)
    return q * s


def fake_quant_affine(x, scale, zp, n_bits: int) -> torch.Tensor:
    qmin, qmax = 0, 2 ** n_bits - 1
    s = scale.clamp(min=1e-8)
    q = (x / s + zp).round().clamp(qmin, qmax)
    return (q - zp) * s


@dataclass
class CalibState:
    method: str
    scale: torch.Tensor
    zero_point: torch.Tensor | None = None
    extra: dict = field(default_factory=dict)


class Observer(ABC):
    name: str = "observer"

    def __init__(self, n_bits: int = BITS_ACT, per_channel: bool = False):
        self.n_bits, self.per_channel = n_bits, per_channel

    @abstractmethod
    def update(self, x: torch.Tensor) -> None: ...

    @abstractmethod
    def finish(self) -> CalibState: ...

    def _reduce_absmax(self, x: torch.Tensor) -> torch.Tensor:
        x = x.detach().float().reshape(-1, x.shape[-1])
        if self.per_channel:
            return x.abs().amax(0)
        return x.abs().amax()

    def _reduce_minmax(self, x: torch.Tensor):
        x = x.detach().float().reshape(-1, x.shape[-1])
        if self.per_channel:
            return x.amin(0), x.amax(0)
        return x.amin(), x.amax()

print("CalibState + Observer base + fake-quant OK")


In [ ]:
class MinMaxObserver(Observer):
    name = "minmax"

    def __init__(self, n_bits=BITS_ACT, per_channel=False):
        super().__init__(n_bits, per_channel)
        self.amax = None

    def update(self, x):
        batch = self._reduce_absmax(x)
        self.amax = batch if self.amax is None else torch.maximum(self.amax, batch.to(self.amax.device))

    def finish(self) -> CalibState:
        a = self.amax.float().clamp(min=1e-8)
        return CalibState(self.name, (a / qmax_sym(self.n_bits)).cpu(), extra={"amax": a.cpu()})


class AffineMinMaxObserver(Observer):
    name = "affine_minmax"

    def __init__(self, n_bits=BITS_ACT, per_channel=False):
        super().__init__(n_bits, per_channel)
        self.vmin = self.vmax = None

    def update(self, x):
        lo, hi = self._reduce_minmax(x)
        self.vmin = lo if self.vmin is None else torch.minimum(self.vmin, lo.to(self.vmin.device))
        self.vmax = hi if self.vmax is None else torch.maximum(self.vmax, hi.to(self.vmax.device))

    def finish(self) -> CalibState:
        lo, hi = self.vmin.float(), self.vmax.float()
        qspan = float(2 ** self.n_bits - 1)
        scale = ((hi - lo) / qspan).clamp(min=1e-8)
        zp = (-lo / scale).round().clamp(0, qspan)
        return CalibState(self.name, scale.cpu(), zp.cpu(), extra={"min": lo.cpu(), "max": hi.cpu()})


class EMAMinMaxObserver(Observer):
    name = "ema_minmax"

    def __init__(self, n_bits=BITS_ACT, per_channel=False, momentum=EMA_MOMENTUM):
        super().__init__(n_bits, per_channel)
        self.momentum, self.amax, self.n = momentum, None, 0

    def update(self, x):
        batch = self._reduce_absmax(x)
        if self.amax is None:
            self.amax = batch
        else:
            b = batch.to(self.amax.device)
            self.amax = self.momentum * self.amax + (1.0 - self.momentum) * b
        self.n += 1

    def finish(self) -> CalibState:
        a = self.amax.float().clamp(min=1e-8)
        return CalibState(self.name, (a / qmax_sym(self.n_bits)).cpu(),
                          extra={"amax": a.cpu(), "n_updates": self.n})

print("MinMax + Affine MinMax + EMA MinMax OK")


In [ ]:
def _take_sample(buf: list[torch.Tensor], cap=SAMPLE_CAP) -> torch.Tensor:
    x = torch.cat(buf, 0) if buf else torch.zeros(1)
    if x.numel() > cap:
        idx = torch.randperm(x.shape[0])[: max(1, cap // max(x.shape[-1], 1))]
        x = x[idx]
    return x.float()


class PercentileObserver(Observer):
    name = "percentile"

    def __init__(self, n_bits=BITS_ACT, per_channel=False, percentile=PERCENTILE):
        super().__init__(n_bits, per_channel)
        self.percentile, self.buf = percentile, []

    def update(self, x):
        self.buf.append(x.detach().float().reshape(-1, x.shape[-1]).cpu())

    def finish(self) -> CalibState:
        x = _take_sample(self.buf)
        q = self.percentile / 100.0
        if self.per_channel:
            a = torch.quantile(x.abs(), q, dim=0)
        else:
            a = torch.quantile(x.abs().reshape(-1), q)
        a = a.clamp(min=1e-8)
        return CalibState(self.name, (a / qmax_sym(self.n_bits)).cpu(),
                          extra={"percentile": self.percentile, "amax": a.cpu()})


class HistogramKLObserver(Observer):
    name = "histogram_kl"

    def __init__(self, n_bits=BITS_ACT, per_channel=False, bins=HIST_BINS):
        super().__init__(n_bits, per_channel)
        self.bins, self.buf = bins, []

    def update(self, x):
        self.buf.append(x.detach().float().reshape(-1).cpu())

    def finish(self) -> CalibState:
        x = torch.cat(self.buf)[:SAMPLE_CAP].float() if self.buf else torch.zeros(1)
        absmax = x.abs().amax().clamp(min=1e-8)
        hist = torch.histc(x, bins=self.bins, min=-absmax.item(), max=absmax.item())
        hist = hist.float() + 1e-8
        n_bins_q = 2 ** self.n_bits
        best_kl, best_i = float("inf"), self.bins
        for i in range(n_bins_q, self.bins + 1, max(1, self.bins // 64)):
            lo = (self.bins - i) // 2
            clipped = hist[lo:lo + i].clone()
            step = i / n_bins_q
            qh = torch.zeros(n_bins_q)
            for k in range(n_bins_q):
                a, b = int(k * step), int((k + 1) * step)
                qh[k] = clipped[a:max(b, a + 1)].sum()
            up = torch.zeros(i)
            for k in range(n_bins_q):
                a, b = int(k * step), int((k + 1) * step)
                span = max(b - a, 1)
                up[a:a + span] = qh[k] / span
            p = clipped / clipped.sum()
            q = (up + 1e-8) / (up.sum() + 1e-8)
            kl = (p * (p / q).log()).sum().item()
            if kl < best_kl:
                best_kl, best_i = kl, i
        clip = absmax * (best_i / self.bins)
        return CalibState(self.name, (clip / qmax_sym(self.n_bits)).cpu(),
                          extra={"kl": best_kl, "clip": clip.cpu()})


class MSEObserver(Observer):
    name = "mse"

    def __init__(self, n_bits=BITS_ACT, per_channel=False, grid=MSE_GRID):
        super().__init__(n_bits, per_channel)
        self.grid, self.buf = grid, []

    def update(self, x):
        self.buf.append(x.detach().float().reshape(-1, x.shape[-1]).cpu())

    def finish(self) -> CalibState:
        x = _take_sample(self.buf)
        amax = x.abs().amax(0) if self.per_channel else x.abs().amax()
        amax = amax.clamp(min=1e-8)
        best_s, best_err = None, float("inf")
        for step in range(self.grid):
            r = 0.5 + 0.5 * (step / max(self.grid - 1, 1))
            s = (amax * r) / qmax_sym(self.n_bits)
            xh = fake_quant_sym(x, s, self.n_bits)
            err = (x - xh).pow(2).mean().item()
            if err < best_err:
                best_err, best_s = err, s
        return CalibState(self.name, best_s.cpu(), extra={"mse": best_err})

print("Percentile + HistogramKL + MSE OK")


## Step 4 — Run all 6 observers on one layer

```
1. Pick one nn.Linear
2. Hook it, run OCR, save X
3. Each observer: update(X) → finish() → fake-quant X
4. Print scale and reconstruction MSE
```

One capture. Six observers. Same hook.


In [ ]:
# Capture input X with a hook
cal = HookCalibrator(model, processor, image, OCR_PROMPT)
cands = [(n, m) for n, m in linear_layers
         if m.weight.numel() >= MIN_PARAMS and not scanner.is_skipped(n)
         and m.in_features <= 1024]
lab_name, lab_layer = (cands[0] if cands else linear_layers[0])
print(f"Lab layer: {lab_name}  W{tuple(lab_layer.weight.shape)}")
print("Hooking input X, running OCR generate...")
captures = cal.collect([lab_name], MAX_CALIB_BATCHES, where="input")
xin = captures[lab_name]
print(f"Captured X: {tuple(xin.shape)}  absmax={xin.abs().max().item():.4g}  "
      f"mean={xin.float().abs().mean().item():.4g}")


In [ ]:
# Run all 6 observers on the same X
OBSERVER_CLASSES = {
    "minmax": MinMaxObserver,
    "affine_minmax": AffineMinMaxObserver,
    "ema_minmax": EMAMinMaxObserver,
    "percentile": PercentileObserver,
    "histogram_kl": HistogramKLObserver,
    "mse": MSEObserver,
}

rows = []
print("Start → end  (same hooked X, 6 observers)\n")
print(f"{'method':<16} {'scale':>12} {'recon MSE':>12}  extra")
print("-" * 72)
x = xin.float()
for name, cls in OBSERVER_CLASSES.items():
    obs = cls(BITS_ACT, per_channel=False)
    obs.update(x)                 # offline: equivalent to observer.update inside the hook
    st = obs.finish()
    scale = st.scale.reshape(-1)[0]
    if st.zero_point is None:
        xh = fake_quant_sym(x, st.scale.to(x.device), BITS_ACT)
    else:
        xh = fake_quant_affine(x, st.scale.to(x.device), st.zero_point.to(x.device), BITS_ACT)
    mse = (x - xh).pow(2).mean().item()
    extra = {k: (float(v) if torch.is_tensor(v) and v.numel() == 1 else v)
             for k, v in st.extra.items() if k in ("percentile", "kl", "mse", "n_updates")}
    print(f"{name:<16} {float(scale):>12.4e} {mse:>12.4e}  {extra}")
    rows.append((name, float(scale), mse, st))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].bar([r[0] for r in rows], [r[1] for r in rows], color="#2980b9")
ax[0].set_ylabel("calibrated scale s"); ax[0].set_title("Activation scale by observer")
ax[0].tick_params(axis="x", rotation=25)
ax[1].bar([r[0] for r in rows], [r[2] for r in rows], color="#27ae60")
ax[1].set_ylabel("||X − Q(X)||²"); ax[1].set_title("Reconstruction MSE on hooked X")
ax[1].tick_params(axis="x", rotation=25)
plt.tight_layout(); plt.show()


### Step 4c — Per-tensor vs per-channel

Same observer (MinMax). Same hook. Different number of scales.

For input `X` with shape tokens × features:

- Per-tensor: one `s` from the global abs-max
- Per-channel: one `s_j` per feature column

A quiet channel is not forced to use a hot channel's scale.


In [ ]:
# Per-tensor vs per-channel MinMax
pt = MinMaxObserver(BITS_ACT, per_channel=False); pt.update(x); s_pt = pt.finish()
pc = MinMaxObserver(BITS_ACT, per_channel=True);  pc.update(x); s_pc = pc.finish()
xh_pt = fake_quant_sym(x, s_pt.scale.to(x.device), BITS_ACT)
xh_pc = fake_quant_sym(x, s_pc.scale.to(x.device), BITS_ACT)
print(f"per-tensor  scale={float(s_pt.scale):.4e}  recon MSE={(x-xh_pt).pow(2).mean().item():.4e}")
print(f"per-channel {s_pc.scale.numel()} scales  "
      f"median s={s_pc.scale.median().item():.4e}  recon MSE={(x-xh_pc).pow(2).mean().item():.4e}")
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(s_pc.scale.numpy(), bins=40, color="#8e44ad")
ax.axvline(float(s_pt.scale), color="#e74c3c", lw=2, label="per-tensor s")
ax.set_xlabel("per-channel scale"); ax.set_title("MinMax granularity")
ax.legend(); plt.tight_layout(); plt.show()


### Step 5 — Input, output, and weight

Same observer (MinMax). Different tensor.

| Tensor | Hook? | Scale is for |
|--------|-------|----------------|
| Input `X` | Yes | activations into the Linear |
| Output `Y` | Yes | activations out of the Linear |
| Weight `W` | No | the weight matrix itself |

`X` and `Y` last only during `forward`. `W` is a parameter, so you read it directly.


In [ ]:
# MinMax on input X, output Y, and weight W
print("Capturing Y (layer output) with the same forward hook...")
ycap = cal.collect([lab_name], max(2, MAX_CALIB_BATCHES // 2), where="output")
yin = ycap[lab_name]

def run_minmax(tensor, label, hooked: bool):
    obs = MinMaxObserver(BITS_ACT, per_channel=False)
    obs.update(tensor)
    st = obs.finish()
    t = tensor.float()
    th = fake_quant_sym(t, st.scale.to(t.device), BITS_ACT)
    mse = (t - th).pow(2).mean().item()
    print(f"  {label:<12} hook={str(hooked):<5}  scale={float(st.scale):.4e}  recon MSE={mse:.4e}  "
          f"absmax={t.abs().max().item():.4g}")
    return float(st.scale), mse

sx, mx = run_minmax(xin, "input X", True)
sy, my = run_minmax(yin, "output Y", True)
sw, mw = run_minmax(lab_layer.weight.detach().float().cpu(), "weight W", False)

fig, ax = plt.subplots(figsize=(5.5, 3.2))
ax.bar(["input X (hook)", "output Y (hook)", "weight W (no hook)"], [sx, sy, sw],
       color=["#2980b9", "#27ae60", "#95a5a6"])
ax.set_ylabel("MinMax scale"); ax.set_title("Same observer, three tensors")
plt.xticks(rotation=15); plt.tight_layout(); plt.show()


### Step 6 — Streaming vs offline

Same observer. Two times to call `update`.

| Schedule | What happens | Memory |
|----------|--------------|--------|
| Streaming | hook calls `update(x)` during `generate` | observer state only |
| Offline | save `X`, then `update(X)` once | full `X` on CPU |

MinMax, Affine MinMax, and EMA MinMax stream naturally. Percentile, Histogram+KL, and MSE need a sample; `collect` is the simple way to get one.


In [ ]:
# Streaming MinMax vs offline collect
stream_obs = MinMaxObserver(BITS_ACT, per_channel=False)
handles = cal.attach([lab_name], {lab_name: stream_obs}, where="input")
cal.run(max(2, MAX_CALIB_BATCHES // 2))
remove_hooks(handles)
st_stream = stream_obs.finish()

off = MinMaxObserver(BITS_ACT, per_channel=False)
off.update(xin)
st_off = off.finish()
print(f"streaming hook  scale={float(st_stream.scale):.6e}  amax={float(st_stream.extra['amax']):.6g}")
print(f"offline collect scale={float(st_off.scale):.6e}  amax={float(st_off.extra['amax']):.6g}")
print("Same method (MinMax). Different only in when update() runs.")


### Step 7 — Real OCR vs Gaussian

Fit MinMax on the hooked OCR `X`, then again on Gaussian noise of the same shape. Score **both** scales on the real `X`. The Gaussian scale does not match OCR range.


In [ ]:
# Real OCR X vs Gaussian, scored on real X
gauss = torch.randn_like(xin)
obs_r, obs_g = MinMaxObserver(BITS_ACT), MinMaxObserver(BITS_ACT)
obs_r.update(xin); obs_g.update(gauss)
sr, sg = obs_r.finish().scale, obs_g.finish().scale
xr = xin.float()
mse_r = (xr - fake_quant_sym(xr, sr, BITS_ACT)).pow(2).mean().item()
mse_g = (xr - fake_quant_sym(xr, sg, BITS_ACT)).pow(2).mean().item()
print(f"{'calib tensor':<18} {'scale':>12} {'MSE on real X':>14}")
print(f"{'real OCR X':<18} {float(sr):>12.4e} {mse_r:>14.4e}")
print(f"{'Gaussian':<18} {float(sg):>12.4e} {mse_g:>14.4e}")
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(["real OCR X", "Gaussian"], [mse_r, mse_g], color=["#27ae60", "#e74c3c"])
ax.set_ylabel("recon MSE of real X"); ax.set_title("Calibrate on the task distribution")
plt.tight_layout(); plt.show()


## Recap

```
document → generate
    hook → X or Y
    observer.update
    finish → scale
    fake-quant to score
```

| Count | Meaning |
|------:|---------|
| 3 | attach points (input, output, weight) |
| 6 | observers (MinMax, Affine MinMax, EMA MinMax, Percentile, Histogram+KL, MSE) |
| 2 | per-tensor or per-channel |
| 2 | streaming hook or offline collect |
